In [1]:
import re
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import pandas as pd
from datetime import datetime

# Weekly Charts

In [2]:
# Target URLs Example
# https://kworb.net/spotify/country/global_weekly.html
# https://kworb.net/spotify/country/us_weekly.html

# Get all Area URLs
def get_all_area_urls():
    url = f"https://kworb.net/spotify/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Define Agent
    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Find required table
    table = soup.select_one("table[style*='width: 410px']")
    if not table:
        return []
    
    # Retrieve all weekly links (Except *_weekly_totals.html)
    results = []
    for tr in table.select("tr"):
        country_td = tr.select_one("td:nth-of-type(1)")
        weekly_a = tr.select_one("a[href$='_weekly.html']")
        country = country_td.get_text(strip=True)
        weekly_url = urljoin(url, weekly_a["href"])

        results.append({"country": country, "url": weekly_url})

    return results

all_area_urls = pd.DataFrame(get_all_area_urls())

In [3]:
# Get Weekly Chart from Certain Area
def get_weekly_chart(url: str, limit=5):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Retrieve update date
    chart_date = None
    title_el = soup.select_one("span.pagetitle")
    if title_el:
        title_text = title_el.get_text(" ", strip=True)
        m = re.search(r"\b(\d{4}/\d{2}/\d{2})\b", title_text)
        if m:
            chart_date = datetime.strptime(m.group(1), "%Y/%m/%d").date()

    # Retrieve Weekly Chart Info
    table = soup.select_one("table#spotifyweekly")
    if not table:
        return []

    chart_data = []
    for row in table.select("tbody tr")[:limit]:
        rank = row.select_one("td:nth-of-type(1)").get_text(strip=True)
        artist = row.select_one("td:nth-of-type(3) a[href*='../artist/']").get_text(strip=True)
        title = row.select_one("td:nth-of-type(3) a[href*='../track/']").get_text(strip=True)
        streams = row.select_one("td:nth-of-type(7)").get_text(strip=True)
        chart_data.append({"rank": rank, "artist": artist, "title": title, "streams": streams, "chart_date": chart_date, "chart_url": url})

    return chart_data

In [4]:
# Scrape all areas weekly charts
def scrape_all_weekly_charts(all_area_urls: pd.DataFrame, limit: int = 20) -> pd.DataFrame:
    all_rows = []
    total = len(all_area_urls)

    # Iterate
    for i, (_, r) in enumerate(all_area_urls.iterrows(), start=1):
        country = r["country"]
        url = r["url"]
        remaining = total - i

        print(f"[{i}/{total}] Scraping {country}... "
              f"({remaining} urls remaining)")

        # Use get_weekly_chart in each area
        chart = get_weekly_chart(url, limit=limit)

        # Add needed infos
        for item in chart:
            item["country"] = country
            all_rows.append(item)

    print("Scraping completed")

    return pd.DataFrame(all_rows)

weekly_chart_df = scrape_all_weekly_charts(all_area_urls, limit=20)

[1/77] Scraping Global... (76 urls remaining)
[2/77] Scraping United States... (75 urls remaining)
[3/77] Scraping United Kingdom... (74 urls remaining)
[4/77] Scraping Andorra... (73 urls remaining)
[5/77] Scraping Argentina... (72 urls remaining)
[6/77] Scraping Australia... (71 urls remaining)
[7/77] Scraping Austria... (70 urls remaining)
[8/77] Scraping Belarus... (69 urls remaining)
[9/77] Scraping Belgium... (68 urls remaining)
[10/77] Scraping Bolivia... (67 urls remaining)
[11/77] Scraping Brazil... (66 urls remaining)
[12/77] Scraping Bulgaria... (65 urls remaining)
[13/77] Scraping Canada... (64 urls remaining)
[14/77] Scraping Chile... (63 urls remaining)
[15/77] Scraping Colombia... (62 urls remaining)
[16/77] Scraping Costa Rica... (61 urls remaining)
[17/77] Scraping Cyprus... (60 urls remaining)
[18/77] Scraping Czech Republic... (59 urls remaining)
[19/77] Scraping Denmark... (58 urls remaining)
[20/77] Scraping Dominican Republic... (57 urls remaining)
[21/77] Scrapin

In [5]:
weekly_chart_df

,rank,artist,title,streams,chart_date,chart_url,country
0,1,Harry Styles,American Girls,"37,500,379",2026-03-12,https://kworb.net/spotify/country/global_weekl...,Global
1,2,PinkPantheress,Stateside + Zara Larsson,"37,407,697",2026-03-12,https://kworb.net/spotify/country/global_weekl...,Global
2,3,Bruno Mars,Risk It All,"35,797,219",2026-03-12,https://kworb.net/spotify/country/global_weekl...,Global
3,4,Bad Bunny,DtMF,"34,350,230",2026-03-12,https://kworb.net/spotify/country/global_weekl...,Global
4,5,Dominic Fike,Babydoll,"31,558,165",2026-03-12,https://kworb.net/spotify/country/global_weekl...,Global
...,...,...,...,...,...,...,...
1535,16,Shartnuss,Nắng có mang em về,"494,984",2026-03-12,https://kworb.net/spotify/country/vn_weekly.html,Vietnam
1536,17,Hngle,Buông,"474,550",2026-03-12,https://kworb.net/spotify/country/vn_weekly.html,Vietnam
1537,18,VCT,Mo,"470,138",2026-03-12,https://kworb.net/spotify/country/vn_weekly.html,Vietnam
1538,19,Sơn Tùng M-TP,Âm Thầm Bên Em,"464,513",2026-03-12,https://kworb.net/spotify/country/vn_weekly.html,Vietnam


In [6]:
#weekly_chart_df[weekly_chart_df["country"] == "Taiwan"]

In [7]:
# Convert streams to float
df = weekly_chart_df.copy()
df['streams'] = df['streams'].str.replace(',', '').astype(float)

In [8]:
streams_df = df.groupby('country')['streams'].sum().reset_index() # sum of streams 
streams_df = streams_df.drop(24, axis=0) # drop global row
streams_df = streams_df.reset_index(drop=True)

In [9]:
tmp = streams_df.sort_values('streams', ascending=False).reset_index()
tmp = tmp.drop('index', axis=1)
tmp

,country,streams
0,United States,165509799.0
1,Mexico,125378363.0
2,Indonesia,119290161.0
3,Brazil,96248886.0
4,India,94795189.0
...,...,...
71,Cyprus,520381.0
72,Iceland,483908.0
73,Luxembourg,360646.0
74,Malta,212784.0


## Country Users Map

In [10]:
import folium
import folium.plugins

In [11]:
# GeoJSON for countries
political_countries_url = ("http://geojson.xyz/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson")

In [ ]:
# Map test (Streams of top 5 songs)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=political_countries_url,
    data=streams_df,
    columns=("country", "streams"),
    key_on="feature.properties.name", # keys to link the data with gejson
    bins=6, 
    fill_color="YlGnBu",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Streams",
    name="Countries by Weekly Total Streams of 20 Songs",
).add_to(m)
folium.LayerControl().add_to(m)

folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)
m.save("streams_map.html")
m



## Artist Country

In [13]:
import re
import requests

In [14]:
def clean_artist_name(name):
    name = re.split(r',|&|feat\.|featuring', name, flags=re.IGNORECASE)[0] # remove features
    return name.strip()

In [15]:
# API using musicbrainz (caution: 2 second rate limit makes many request slow) 
def get_artist_country(artist_name):

    artist_name = clean_artist_name(artist_name)
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    url = "https://musicbrainz.org/ws/2/artist/"
    params = {"query": artist_name, 
              "fmt": "json",
              "limit": 1}

    try:
        time.sleep(2.0)
        response = requests.get(url, params=params, headers=headers)
        data = response.json()

        if data['artists']:
            artist = data['artists'][0]

            if 'country' in artist: # country
                return artist['country']

            area = artist.get('area')
            if isinstance(area, dict):
                return area.get('name', 'Unknown')

    except:
        pass

    return "Unknown"

In [16]:
get_artist_country('Bad Bunny') # test

'PR'

In [17]:
# artist country dictionary
# try for weekly_chart_df
unique_artists = weekly_chart_df['artist'].unique()

artist_country = {}
for artist in unique_artists:
    artist_country[artist] = get_artist_country(artist)

In [18]:
#artist_country

Next we can try for top 500 artists

In [19]:
# top 500 artist streams (total) 
def get_artist_top500(url: str, limit=500):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    html = requests.get(url, headers=headers, timeout=20)
    html.raise_for_status()
    soup = BeautifulSoup(html.content, "html.parser")

    # Retrieve Chart Info
    table = soup.select_one("div.container table")
    if not table:
        return []

    chart_data = []
    rows = table.select("tr")
    for row in table.select("tbody tr")[:limit]:
        artist = row.select_one("td:nth-of-type(1)").get_text(strip=True)
        streams = row.select_one("td:nth-of-type(2)").get_text(strip=True)
        chart_data.append({"artist": artist, "total_streams": streams, "chart_url": url})

    chart_data = pd.DataFrame(chart_data)
    return chart_data 

In [20]:
url = "https://kworb.net/spotify/artists.html"
top500_artist_df = get_artist_top500(url, limit=500)

In [21]:
top500_artist_df

,artist,total_streams,chart_url
0,Drake,"129,424.5",https://kworb.net/spotify/artists.html
1,Taylor Swift,"121,961.5",https://kworb.net/spotify/artists.html
2,Bad Bunny,"118,262.1",https://kworb.net/spotify/artists.html
3,The Weeknd,"91,402.1",https://kworb.net/spotify/artists.html
4,Justin Bieber,"72,217.8",https://kworb.net/spotify/artists.html
...,...,...,...
495,Bibi und Tina,"6,588.0",https://kworb.net/spotify/artists.html
496,Outkast,"6,578.9",https://kworb.net/spotify/artists.html
497,Los Temerarios,"6,559.2",https://kworb.net/spotify/artists.html
498,The Cure,"6,554.5",https://kworb.net/spotify/artists.html


In [22]:
# Country dictionary (caution: takes a long time to run)
artist_country = {}
for artist in top500_artist_df['artist']:
    artist_country[artist] = get_artist_country(artist)

In [23]:
#artist_country 

In [24]:
top500_artist_df['country'] = top500_artist_df['artist'].map(artist_country)

In [25]:
top500_artist_df['country'].unique()

<ArrowStringArray>
[                        'CA',                         'US',
                         'PR',                         'GB',
                         'CO',                         'KR',
                         'FR',                         'IN',
                   'Scotland',                         'AU',
                         'MX',                    'England',
                         'DE',                         'SE',
                         'NO',                'Los Angeles',
                         'PA',                         'NL',
                         'BR',                         'JM',
                         'SN',                         'IE',
               'Buenos Aires',                         'AR',
                         'ES', 'Las Palmas de Gran Canaria',
                    'McAllen',                    'Sinaloa',
                    'Unknown',                'Guadalajara',
                      'Miami',                         'NZ',
     

In [26]:
# What are the unknowns
top500_artist_df.loc[top500_artist_df['country'] == 'Unknown']

,artist,total_streams,chart_url,country
216,Mora,"12,541.9",https://kworb.net/spotify/artists.html,Unknown
298,League of Legends,"10,112.7",https://kworb.net/spotify/artists.html,Unknown
494,Original Broadway Cast of Hamilton,"6,645.8",https://kworb.net/spotify/artists.html,Unknown
495,Bibi und Tina,"6,588.0",https://kworb.net/spotify/artists.html,Unknown


In [27]:
# change names to countries

# https://www.iban.com/country-codes (ISO 3166-1 Alpha-2 country codes)
country_rename = {"Scotland": "GB", 
                  "Buenos Aires": "AR", 
                  "McAllen": "US",
                  "Miami": "US",
                  "Hawaii": "US",
                  "Gujarat": "IN", 
                  "England": "GB",
                  "Los Angeles": "US",
                  "Las Palmas de Gran Canaria": "ES",
                  "Sinaloa": "MX",
                  "Guadalajara": "MX",
                  "Mazatlan": "MX",
                  "Culiacán": "MX", 
                  "Punjab": "IN"}

top500_artist_new = top500_artist_df.copy()
top500_artist_new['country'] = top500_artist_new['country'].replace(country_rename)

# Manually change unknowns
top500_artist_new.loc[top500_artist_new['artist'] == 'Mora', 'country'] = 'PR'
top500_artist_new.loc[top500_artist_new['artist'] == 'League of Legends', 'country'] = 'US'
top500_artist_new.loc[top500_artist_new['artist'] == 'Original Broadway Cast of Hamilton', 'country'] = 'US'
top500_artist_new.loc[top500_artist_new['artist'] == 'Bibi und Tina', 'country'] = 'DE'

In [28]:
top500_artist_new['country'].unique()

<ArrowStringArray>
['CA', 'US', 'PR', 'GB', 'CO', 'KR', 'FR', 'IN', 'AU', 'MX', 'DE', 'SE', 'NO',
 'PA', 'NL', 'BR', 'JM', 'SN', 'IE', 'AR', 'ES', 'NZ', 'CL', 'NG', 'IT', 'DO',
 'BE', 'AT', 'GT', 'PK', 'DK']
Length: 31, dtype: str

In [29]:
#pd.set_option('display.max_rows', None)
#pd.reset_option('display.max_rows')
top500_artist_new

,artist,total_streams,chart_url,country
0,Drake,"129,424.5",https://kworb.net/spotify/artists.html,CA
1,Taylor Swift,"121,961.5",https://kworb.net/spotify/artists.html,US
2,Bad Bunny,"118,262.1",https://kworb.net/spotify/artists.html,PR
3,The Weeknd,"91,402.1",https://kworb.net/spotify/artists.html,CA
4,Justin Bieber,"72,217.8",https://kworb.net/spotify/artists.html,CA
...,...,...,...,...
495,Bibi und Tina,"6,588.0",https://kworb.net/spotify/artists.html,DE
496,Outkast,"6,578.9",https://kworb.net/spotify/artists.html,US
497,Los Temerarios,"6,559.2",https://kworb.net/spotify/artists.html,MX
498,The Cure,"6,554.5",https://kworb.net/spotify/artists.html,GB


In [30]:
top500_artist_countries_count = top500_artist_new['country'].value_counts().reset_index()
top500_artist_countries_count.columns = ['country', 'count']

In [31]:
# tmp = top500_artist_countries_count
# tmp['country'] = tmp['country'].apply(convert_country)

In [32]:
# tmp

In [33]:
type(top500_artist_countries_count['country'])

pandas.Series

In [34]:
# GeoJSON for countries
countries_2code_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
countries_2code_json = requests.get(countries_2code_url).json()

In [35]:
countries_2code_json["features"][0]["properties"]

{'name': 'Indonesia', 'ISO3166-1-Alpha-3': 'IDN', 'ISO3166-1-Alpha-2': 'ID'}

In [36]:
countries_2code_json.keys()

dict_keys(['type', 'name', 'crs', 'features'])

In [ ]:
# Map (Top 500 artists by country)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=countries_2code_json,
    data=top500_artist_countries_count,
    columns=("country", "count"),
    key_on="properties.ISO3166-1-Alpha-2", # match with the alpha2
    threshold_scale=[1, 5, 10, 30, 50, 245],
    fill_color="YlGnBu",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Number of Artists",
    name="Countries by Number of Artists in Top 500 Streams",
).add_to(m)
folium.LayerControl().add_to(m)

folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)

fig.save('artists_map.html')
m



## Artist Genres

In [38]:
# API using musicbrainz (caution: 2 second rate limit makes many request slow) 
def get_artist_genre(artist_name):

    artist_name = clean_artist_name(artist_name)
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    url = "https://musicbrainz.org/ws/2/artist/"
    params = {"query": artist_name, 
              "fmt": "json",
              "limit": 1}

    try:
        time.sleep(2.0)
        response = requests.get(url, params=params, headers=headers)
        data = response.json()

        if data['artists']:
            artist = data['artists'][0]

            # music brainz has genres entity so try it out
            genres = artist.get('genres', [])
            if genres:
                # users tag genres to artist so sort by count genre count and take the most common one
                genres_sorted = sorted(genres, key=lambda x: x["count"], reverse=True)
                return genres_sorted[0]["name"].title()

            # many unknowns, try same methodology for "tags" entity
            tags = artist.get("tags", [])
            if tags:
                tags_sorted = sorted(tags, key=lambda x: x["count"], reverse=True)
                return tags_sorted[0]["name"].title()

    except:
        pass

    return "Unknown"

In [39]:
# test
get_artist_genre('Bad Bunny')

'Reggaeton'

In [40]:
# apply to our df (caution: takes a long time to run)
artist_genre = {} # dictionary
for artist in top500_artist_df['artist']:
    artist_genre[artist] = get_artist_genre(artist)

In [41]:
#pd.set_option('display.max_rows', None)
#pd.reset_option('display.max_rows')
top500_artist_new['genre'] = top500_artist_new['artist'].map(artist_genre)
top500_artist_new

,artist,total_streams,chart_url,country,genre
0,Drake,"129,424.5",https://kworb.net/spotify/artists.html,CA,Hip Hop
1,Taylor Swift,"121,961.5",https://kworb.net/spotify/artists.html,US,Pop
2,Bad Bunny,"118,262.1",https://kworb.net/spotify/artists.html,PR,Reggaeton
3,The Weeknd,"91,402.1",https://kworb.net/spotify/artists.html,CA,R&B
4,Justin Bieber,"72,217.8",https://kworb.net/spotify/artists.html,CA,Pop
...,...,...,...,...,...
495,Bibi und Tina,"6,588.0",https://kworb.net/spotify/artists.html,DE,Series Title As Artist
496,Outkast,"6,578.9",https://kworb.net/spotify/artists.html,US,Hip Hop
497,Los Temerarios,"6,559.2",https://kworb.net/spotify/artists.html,MX,Unknown
498,The Cure,"6,554.5",https://kworb.net/spotify/artists.html,GB,New Wave


In [42]:
# What are the unknowns
#top500_artist_new.loc[top500_artist_new['genre'] == 'Unknown']

In [43]:
print(top500_artist_new['genre'].unique())

<ArrowStringArray>
[         'Hip Hop',              'Pop',        'Reggaeton',
              'R&B',  'Alternative Pop',            'K-Pop',
 'Alternative Rock',        'Dance-Pop',            'House',
        'Indie Pop',
 ...
      'Bedroom Pop',            'Metal',       'Jangle Pop',
           'Cumbia',            'Forró',      'Heavy Metal',
        'Folk Rock',           'Lgbtqi',           'Indian',
  'Special Purpose']
Length: 111, dtype: str


In [44]:
top500_artist_genre_count = top500_artist_new['genre'].value_counts().reset_index()
top500_artist_genre_count.columns = ['genre', 'count']

In [45]:
#top500_artist_genre_count

In [46]:
# remove unknown
top_genres = top500_artist_genre_count[top500_artist_genre_count['genre'] != 'Unknown']
top_genres = top_genres.head(20)
top_genres

,genre,count
0,Hip Hop,86
1,Pop,72
2,Reggaeton,25
4,Latin,19
5,Rock,15
6,Contemporary R&B,11
7,Alternative Rock,10
8,K-Pop,9
9,Indie Pop,9
10,Country,9


In [47]:
# create manual groupings
genre_map = {'Hip Hop': 'Hip Hop', 'Trap': 'Hip Hop', 'Pop Rap': 'Hip Hop',
    'Pop': 'Pop', 'Indie Pop': 'Pop', 'K-Pop': 'Pop',
    'Electronic': 'Electronic', 'Dance-Pop': 'Electronic', 'House': 'Electronic',
    'Latin': 'Latin', 'Reggaeton': 'Latin', 'Trap Latino': 'Latin', 'Latin Pop': 'Latin', 'Regional Mexicano': 'Latin',
    'Rock': 'Rock', 'Indie Rock': 'Rock', 'Alternative Rock': 'Rock',
    'Country': 'Other/Region Genre', 'Filmi': 'Other/Region Genre', 'Contemporary R&B': 'Other/Region Genre'
}
             
            
top_genres['genre_main'] = top_genres['genre'].map(genre_map)

In [ ]:
# Preliminary plot
import plotly.express as px

color_map = {'Hip Hop': '#ff0000',
             'Pop': '#0000ff',
             'Electronic':'#ff00ff',
             'Latin': '#ffff00',
             'Rock': '#00ff00',
             'Other/Region Genre': '#00ffff'           
}  

fig = px.bar(top_genres, 
    x='count', y='genre', title="Top 20 Genres Represented By Top 500 Global Artists",
    color = 'genre_main', color_discrete_map = color_map, orientation='h'
)

fig.update_layout(yaxis={'categoryorder': 'total ascending'},
    height = 600, plot_bgcolor='rgba(0,0,0,0)')

top_genres = top_genres[top_genres['genre'] != 'Global'] # remove global

fig.show()
fig.write_html("top_genres.html")

# Distribution Across Countries

Lets look at top 20 for each country

In [49]:
weekly_chart_df_20 = scrape_all_weekly_charts(all_area_urls, limit=20)

[1/77] Scraping Global... (76 urls remaining)
[2/77] Scraping United States... (75 urls remaining)
[3/77] Scraping United Kingdom... (74 urls remaining)
[4/77] Scraping Andorra... (73 urls remaining)
[5/77] Scraping Argentina... (72 urls remaining)
[6/77] Scraping Australia... (71 urls remaining)
[7/77] Scraping Austria... (70 urls remaining)
[8/77] Scraping Belarus... (69 urls remaining)
[9/77] Scraping Belgium... (68 urls remaining)
[10/77] Scraping Bolivia... (67 urls remaining)
[11/77] Scraping Brazil... (66 urls remaining)
[12/77] Scraping Bulgaria... (65 urls remaining)
[13/77] Scraping Canada... (64 urls remaining)
[14/77] Scraping Chile... (63 urls remaining)
[15/77] Scraping Colombia... (62 urls remaining)
[16/77] Scraping Costa Rica... (61 urls remaining)
[17/77] Scraping Cyprus... (60 urls remaining)
[18/77] Scraping Czech Republic... (59 urls remaining)
[19/77] Scraping Denmark... (58 urls remaining)
[20/77] Scraping Dominican Republic... (57 urls remaining)
[21/77] Scrapin

In [50]:
# Map artists to country (caution: takes a long time to run)
unique_artists = weekly_chart_df_20['artist'].unique()
artist_country2 = {}
for artist in unique_artists:
    artist_country2[artist] = get_artist_country(artist)

In [51]:
# Map artists to genre (caution: takes a long time to run)
artist_genre2 = {}
for artist in unique_artists:
    artist_genre2[artist] = get_artist_genre(artist)

In [52]:
# apply dictionaries to df
weekly_chart_df_20['artist_country'] = weekly_chart_df_20['artist'].map(artist_country2)
weekly_chart_df_20['artist_genre'] = weekly_chart_df_20['artist'].map(artist_genre2)

In [53]:
#pd.reset_option("display.max_rows")
unique_country = weekly_chart_df_20.loc[weekly_chart_df_20['artist_country'] != 'Unknown', 'artist_country'].drop_duplicates()
unique_country

0         England
2              US
3              PR
4         Florida
6              GB
          ...    
1522           BE
1523           VN
1525    Bắc Giang
1529     Montréal
1534        Hanoi
Name: artist_country, Length: 127, dtype: str

In [54]:
len(unique_country)

127

In [55]:
# check unknowns
unique_unknowns = weekly_chart_df_20.loc[weekly_chart_df_20['artist_country'] == 'Unknown', 'artist'].drop_duplicates()
len(unique_unknowns)

119

In [56]:
# change names to countries

# https://www.iban.com/country-codes (ISO 3166-1 Alpha-2 country codes)
country_rename = {"Scotland": "GB", "Buenos Aires": "AR", "McAllen": "US", "Miami": "US", "Hawaii": "US",
                  "Gujarat": "IN", "England": "GB", "Los Angeles": "US", "Las Palmas de Gran Canaria": "ES",
                  "Sinaloa": "MX", "Guadalajara": "MX", "Mazatlan": "MX", "Culiacán": "MX", "Punjab": "IN",
                  "Florida": "US", "Montgomery": "US", "Lübeck": "DE", "Santiago": "CL", "San Francisco": "US",
                  "Barcelona": "ES", "Karlovy Vary": "CZ", "Copenhagen": "DK", "Helsinki": "FI", "Athens": "GR",
                  "Mumbai": "IN", "Uttarakhand": "IN", "Jammu and Kashmir": "IN", "Genova": "CH", "Napoli": "IT",
                  "Moscow": "RU", "New York": "US", "Paris": "FR", "El Jadida": "MA", "Lagos": "NG", "Port Elizabeth": "ZA",
                  "Nes": "NO", "Nes": "NO", "Karachi": "PK", "Liverpool": "UK", "Davao City": "PH", "Tacloban": "PH", 
                  "Szczecin": "PL", "Warsaw": "PL", "Brooklyn": "US", "Ciudad de México": "MX", "Sevilla": "ES", 
                  "Kaohsiung": "TW", "Kyïv": "UA", "Kerala": "IN", "Montréal": "CA", "Hanoi": "VN", "Ho Chi Minh": "VN",
                  "Lahore": "PK", "Ski": "NO", "Ufa": "RU"}

weekly_chart_df_20_new = weekly_chart_df_20.copy()
weekly_chart_df_20_new['artist_country'] = weekly_chart_df_20_new['artist_country'].replace(country_rename)

In [57]:
# check
unique_country = weekly_chart_df_20_new.loc[weekly_chart_df_20['artist_country'] != 'Unknown', 'artist_country'].drop_duplicates()
#unique_country 

In [58]:
# check unknowns
unique_unknown_country = weekly_chart_df_20_new.loc[weekly_chart_df_20_new['artist_country'] == 'Unknown', 'artist'].drop_duplicates()
unique_unknown_genre = weekly_chart_df_20_new.loc[weekly_chart_df_20_new['artist_genre'] == 'Unknown', 'artist'].drop_duplicates()
#unique_unknown_country

The unknowns are for a number of reasons: 
1. user not in database
2. user in database but no associated country/area codes
3. database name and charts name are not matching in language
4. Name overlaps causes database to pick wrong artist.

In [59]:
len(unique_unknown_country)

119

In [60]:
len(unique_unknown_genre)

303

In [61]:
len(unique_artists)

589

In [62]:
len(unique_country)

86

In [63]:
151/581

0.25989672977624784

In [64]:
300/581

0.5163511187607573

# Listener Distribution Among Countries

Do people listen to artists from domestic or international?

In [65]:
import pycountry

# convert country names to iso code
def convert_iso(country_name):

    try:
        return pycountry.countries.lookup(country_name).alpha_2
    except: 
        return 'Unknown'

In [66]:
# convert iso code to country names (will use later for plots)
def convert_country(iso_code):
    try:
        return pycountry.countries.get(alpha_2=iso_code.upper()).name
    except: 
        return 'Unknown'

In [67]:
# get iso codes
weekly_chart_df_20_new["country_iso"] = weekly_chart_df_20_new["country"].apply(convert_iso)

In [68]:
#weekly_chart_df_20_new

In [69]:
type(weekly_chart_df_20_new)

pandas.DataFrame

In [70]:
weekly_chart_df_20_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 1540 entries, 0 to 1539
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   rank            1540 non-null   str   
 1   artist          1540 non-null   str   
 2   title           1540 non-null   str   
 3   streams         1540 non-null   str   
 4   chart_date      1540 non-null   object
 5   chart_url       1540 non-null   str   
 6   country         1540 non-null   str   
 7   artist_country  1540 non-null   str   
 8   artist_genre    1540 non-null   str   
 9   country_iso     1540 non-null   str   
dtypes: object(1), str(9)
memory usage: 271.5+ KB


In [71]:
# Define song as domestic, international, or unknown
def artist_locality(row):
    artist_country = row['artist_country']
    chart_country = row['country_iso']

    if artist_country == 'Unknown':
        return 'Unknown'
        
    if artist_country == chart_country:
        return 'Domestic'
    else:
        return 'International'

In [72]:
weekly_chart_df_20_new["artist_locality"] = weekly_chart_df_20_new.apply(artist_locality, axis=1)

## Artist Locality Percentage Share

In [73]:
# Change streams to integer from string
weekly_chart_df_20_new['streams'] = weekly_chart_df_20_new['streams'].str.replace(',', '').astype(int)

In [74]:
# get streams df
streams_df = weekly_chart_df_20_new.groupby(['country', 'artist_locality'])['streams'].sum().unstack(fill_value=0)
streams_df

artist_locality,Domestic,International,Unknown
country,,,
Andorra,0,50614,0
Argentina,4151128,17040004,2789854
Australia,0,24915633,1813772
Austria,0,3481086,0
Belarus,0,596818,246470
...,...,...,...
United Kingdom,44990847,9685489,4544552
United States,40936774,118224124,6348901
Uruguay,81211,1735183,258503


In [75]:
streams_df['total_streams'] = streams_df.sum(axis=1) # total number of streams
#streams_df

In [76]:
# percentages for each category
streams_df['domestic_pct'] = (streams_df['Domestic'] / streams_df['total_streams']) * 100
streams_df['int_pct'] = (streams_df['International'] / streams_df['total_streams']) * 100
streams_df['unknown_pct'] = (streams_df['Unknown'] / streams_df['total_streams']) * 100

In [77]:
streams_df.head(10)

artist_locality,Domestic,International,Unknown,total_streams,domestic_pct,int_pct,unknown_pct
country,,,,,,,
Andorra,0,50614,0,50614,0.000000,100.000000,0.000000
Argentina,4151128,17040004,2789854,23980986,17.310081,71.056311,11.633608
Australia,0,24915633,1813772,26729405,0.000000,93.214320,6.785680
Austria,0,3481086,0,3481086,0.000000,100.000000,0.000000
Belarus,0,596818,246470,843288,0.000000,70.772737,29.227263
Belgium,0,5614286,254140,5868426,0.000000,95.669367,4.330633
Bolivia,0,1772899,262882,2035781,0.000000,87.086921,12.913079
Brazil,42425203,17715261,36108422,96248886,44.078643,18.405679,37.515678
Bulgaria,373646,605454,222326,1201426,31.100209,50.394614,18.505176


In [78]:
streams_df = streams_df.drop('Global')
streams_df = streams_df.reset_index()

In [79]:
domestic_sort = streams_df.sort_values('domestic_pct', ascending=True)
int_sort = streams_df.sort_values('int_pct', ascending=False)
unknown_sort = streams_df.sort_values('unknown_pct', ascending=False)

#domestic_sort.head(5) #--> 100% highest
#int_sort.head(5)# --> 100% highest
#unknown_sort.head(5)# --> 79% highest

In [ ]:

fig = px.bar(domestic_sort, 
    x=["domestic_pct", "int_pct", "unknown_pct"], y="country", title="Music Consumption by Artist Locality by Country",
    labels={"value": "Percentage (%)", "country": "Country", "variable": "Locality"},
    color_discrete_map={"domestic_pct": "skyblue", "int_pct": "#50C878", "unknown_pct": "grey"},
    orientation='h', height=1500 # Tall height so country names are readable
)

fig.update_layout(
    barmode='stack',
    legend_title_text='Category',
    xaxis_range=[0, 100], # Force 0 to 100%
    hovermode="y unified" # Shows all three values when hovering over a country
)

fig.show()
fig.write_html("artist_locality_interactive.html")

## Manually update country/genre for top artists

In [81]:
# Country
top_unknowns = weekly_chart_df_20_new[weekly_chart_df_20_new['artist_country'] == 'Unknown']['artist'].value_counts()
#print(top_unknowns[top_unknowns > 1])

In [82]:
# Manually change top unknowns
update = {
    'Omar Courtz': {'country': 'PR', 'genre': 'Reggaeton'},
    'El Bogueto': {'country': 'MX', 'genre': 'Reggaeton'},
    'HUNTR/X': {'country': 'US', 'genre': 'K-Pop'},
    'Jin': {'country': 'KR', 'genre': 'K-Pop'},
    'HermesHermes': {'country': 'GR', 'genre': 'Hip Hop'},
    'DJ Japa NK': {'country': 'BR', 'genre': 'Funk'},
    'W Sound': {'country': 'AR', 'genre': 'Trap Latino'},
    'V': {'country': 'KR', 'genre': 'K-Pop'},
    'Bella Kay' : {'country': 'US', 'genre': 'Indie Pop'},
    'Mr Plata': {'country': 'CO', 'genre': 'Reggaeton'},
    'Lvbel C5': {'country': 'TR', 'genre': 'Hip Hop'},
    'урал гайсин': {'country': 'RU', 'genre': 'K-Pop'},
    'ARIA VEGA': {'country': 'CO', 'genre': 'R&B'},
    'TUL8TE': {'country': 'EG', 'genre': 'Hip Hop'},
    'Ursaru': {'country': 'RO', 'genre': 'Hip Hop'},
    'Aarne': {'country': 'RO', 'genre': 'Hip Hop'},
    'dabbackwood': {'country': 'RU', 'genre': 'Plugg'},
    'Rambo goyard': {'country': 'FR', 'genre': 'Hip Hop'},
    'Selina': {'country': 'BG', 'genre': 'Pop'},
    'Emanuela': {'country': 'BG', 'genre': 'Chalga'},
    'Mirela': {'country': 'BG', 'genre': 'Pop'},
    'Noah Kahan': {'country': 'US', 'genre': 'Folk Pop'},
    'Jere Klein': {'country': 'CL', 'genre': 'Reggaeton'},
    'Yeison Jimenez': {'country': 'CO', 'genre': 'K-Pop'},
    'YOVNGCHIMI': {'country': 'PR', 'genre': 'Trap Latino'},
    'Çağla': {'country': 'TR', 'genre': 'Pop'},
    'Poizi': {'country': 'TR', 'genre': 'Hip Hop'},
    '4rano': {'country': 'SK', 'genre': 'Pop'},
    'ZIAD ZAZA': {'country': 'EG', 'genre': 'Hip Hop'},
    'La Rvfleuze': {'country': 'FR', 'genre': 'Hip Hop'},
    'Trannos': {'country': 'GR', 'genre': 'Hip Hop'},
    'Navjot Ahuja': {'country': 'IN', 'genre': 'Pop'},
    'Idgitaf': {'country': 'ID', 'genre': 'Indonesian Pop'},
    'Raim Laode': {'country': 'ID', 'genre': 'Indonesian Pop'},
    'Kid Yugi': {'country': 'IT', 'genre': 'Hip Hop'},
    'Ernar Amandyq': {'country': 'KZ', 'genre': 'Indie-Pop'},
    'Mavo': {'country': 'NG', 'genre': 'Afrobeats'},
    'FOLA': {'country': 'NG', 'genre': 'Afrobeats'},
    'fitterkarma': {'country': 'PH', 'genre': 'Rock'},
    'Feza': {'country': 'ZA', 'genre': 'Maskandi'},
    'Ntencane': {'country': 'ZA', 'genre': 'Maskandi'},
    'Sam Deep': {'country': 'ZA', 'genre': 'House'},  
    'BLOK3': {'country': 'TR', 'genre': ' Hip Hop'},
    'Jimin': {'country': 'KR', 'genre': 'K-Pop'},
    'Ezzy R': {'country': 'DR', 'genre': 'Hip Hop'}
}

weekly_chart_df_20_clean = weekly_chart_df_20_new.copy()

for artist, info in update.items():
    weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist'] == artist, ['artist_country', 'artist_genre']] = [info['country'], info['genre']]

In [83]:
# check unknowns
unique_unknowns_country = weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_country'] == 'Unknown', 'artist'].drop_duplicates()

In [84]:
len(unique_unknowns_country)/len(unique_artists)

0.1494057724957555

In [85]:
pd.set_option('display.max_rows', None)
genre_count = weekly_chart_df_20_clean['artist_genre'].value_counts().reset_index()
genre_count.columns = ['genre', 'count']



In [86]:
# Genre
top_unknowns = weekly_chart_df_20_clean[weekly_chart_df_20_clean['artist_genre'] == 'Unknown']['artist'].value_counts()
print(top_unknowns[top_unknowns > 2])

artist
Omer Adam           9
Hafdís Huld         8
Beton.Hofi          7
shadowraze          7
Rels B              6
Lege-Cy             6
Disco Lines         5
madk1d              5
RnBoi               5
Karan Aujla         5
Nadhif Basalamah    5
Lauta               4
SADU                4
Ares                4
YOUNGOHM            4
Nono La Grinta      3
Milo j              3
Eypio               3
triibupasta         3
Shashwat Sachdev    3
ElGrandeToto        3
Ekipa               3
Sentino             3
HANRORO             3
Drevo               3
HIEUTHUHAI          3
Name: count, dtype: int64


In [87]:
# Fix DR genre
weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist'] == 'Ronny GTA', 'artist_genre'] = 'Latin Urban'


In [88]:
# Fix HK genre
weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_genre'] == 'Actor', 'artist_genre'] = 'Cantopop'

In [89]:
unique_unknowns_genre = weekly_chart_df_20_clean.loc[weekly_chart_df_20_clean['artist_genre'] == 'Unknown', 'artist'].drop_duplicates()

In [90]:
len(unique_unknowns_genre)/len(unique_artists)

0.4617996604414261

## Redo counts with cleaned df

In [91]:
weekly_chart_df_20_clean["artist_locality"] = weekly_chart_df_20_clean.apply(artist_locality, axis=1)

In [92]:
# get streams df
streams_df2 = weekly_chart_df_20_clean.groupby(['country', 'artist_locality'])['streams'].sum().unstack(fill_value=0)

In [93]:
streams_df2['total_streams'] = streams_df2.sum(axis=1) # total number of streams

In [94]:
# percentages for each category
streams_df2['domestic_pct'] = (streams_df2['Domestic'] / streams_df2['total_streams']) * 100
streams_df2['int_pct'] = (streams_df2['International'] / streams_df2['total_streams']) * 100
streams_df2['unknown_pct'] = (streams_df2['Unknown'] / streams_df2['total_streams']) * 100

In [95]:
streams_df2 = streams_df2.reset_index()

In [96]:
domestic_sort2 = streams_df2.sort_values('domestic_pct', ascending=True)
int_sort2 = streams_df2.sort_values('int_pct', ascending=False)
unknown_sort2 = streams_df2.sort_values('unknown_pct', ascending=False)

#domestic_sort.head(5) #--> 100% highest
#int_sort.head(5)# --> 100% highest
#unknown_sort.head(5)# --> 33% highest

In [ ]:
import plotly.express as px

fig = px.bar(domestic_sort2, 
    x=["domestic_pct", "int_pct", "unknown_pct"], y="country", title="Music Consumption by Artist Locality by Country Cleaned Dataset",
    labels={"value": "Percentage (%)", "country": "Country", "variable": "Locality"},
    color_discrete_map={"domestic_pct": "skyblue", "int_pct": "#50C878", "unknown_pct": "grey"},
    orientation='h', height=1500 # Tall height so country names are readable
)

fig.update_layout(
    barmode='stack',
    legend_title_text='Category',
    xaxis_range=[0, 100], # Force 0 to 100%
    hovermode="y unified" # Shows all three values when hovering over a country
)

fig.show()
fig.write_html("artist_locality_interactive_clean.html")

## Domestic vs International listening Map


In [98]:
streams_df2["country_iso"] = streams_df2["country"].apply(convert_iso) # get iso codes

In [99]:
# manually fix Russia and Turkey
streams_df2.loc[streams_df2['country'] == 'Russia', 'country_iso'] = 'RU'
streams_df2.loc[streams_df2['country'] == 'Turkey', 'country_iso'] = 'TR'

In [ ]:
# Map (Domestic Percentage)
m = folium.Map(location=(30, 10), zoom_start=2, tiles="cartodb positron")
folium.Choropleth(
    geo_data=countries_2code_json, # same json as before
    data=streams_df2,
    columns=("country_iso", "domestic_pct"),
    key_on="properties.ISO3166-1-Alpha-2", # match with the alpha2
    fill_color="GnBu",
    fill_opacity=1.0,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Percentage of Top 20 Streams By Domestic Artists",
    name="Countries by Number of Artists in Top 500 Streams",
).add_to(m)


fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)

fig.save("domestic_artists_map.html")
m



## Web Plot

In [101]:
# remove global
weekly_chart_df_20_clean = weekly_chart_df_20_clean[weekly_chart_df_20_clean['country'] != 'Global'] 

In [102]:
# get country from ISO
weekly_chart_df_20_clean["artist_country_full"] = (weekly_chart_df_20_clean["artist_country"].apply(convert_country))

In [103]:
# Manually change top unknowns
update = {'Korea, Republic of': 'South Korea', 
          'Russian Federation': 'Russia',
          'Türkiye': 'Turkey',
          'Moldova, Republic of': 'Moldova',
          'Taiwan, Province of China': 'Taiwan',
          'Viet Nam': 'Vietnam',
          'Venezuela, Bolivarian Republic of': 'Venezuela'
}

web_df = weekly_chart_df_20_clean.copy()
web_df['artist_country_full'] = web_df['artist_country_full'].replace(update)


In [104]:
# web_df

In [ ]:
from pyvis.network import Network
# https://pyvis.readthedocs.io/en/latest/

g = Network(height="800px", width="100%", notebook=True,bgcolor="white")

# add country nodes
countries = set(web_df["country"]).union(set(web_df["artist_country_full"]))
stream_totals = (web_df.groupby("country")["streams"].sum())

for c in countries:
    size = stream_totals.get(c, 1) / 1.8e6 # scale size by streams
    g.add_node(c, label=c, size=size, font={'size': 22})
    

# aggregate stream flows
flows = (
    web_df[web_df["artist_locality"] == "International"]
    .groupby(["country", "artist_country_full"])["streams"]
    .sum()
    .reset_index()
)

# add edges
for _, row in flows.iterrows():
    g.add_edge(
        row["country"],
        row["artist_country_full"],
        value=row["streams"]
    )

g.show("music_flows.html")


## Genre Map

In [106]:
# group by genre and country
genre_df = (weekly_chart_df_20_clean.groupby(['country', 'artist_genre'])['streams'].sum().reset_index())
genre_df = genre_df[genre_df['artist_genre'] != 'Unknown']

In [107]:
# obtain relative stream share by genre per country
genre_df['stream_pct'] = genre_df['streams'] / genre_df.groupby('country')['streams'].transform('sum') 


In [108]:
# obtain highest
genre_df_first = (genre_df.sort_values('stream_pct', ascending=False).groupby('country').first()).reset_index()

In [109]:
genre_df_first['artist_genre'].value_counts()

artist_genre
Pop                  22
Reggaeton            17
Hip Hop              10
K-Pop                 5
Funk                  2
Regional Mexicano     2
Chalga                1
Latin Urban           1
Indietronica          1
Cantopop              1
Lo-Fi Hip Hop         1
Filmi                 1
Indonesian Pop        1
Psytrance             1
J-Rock                1
Indie-Pop             1
French                1
Neo Soul              1
Afrobeats             1
Singer-Songwriter     1
Rock                  1
Alternative Pop       1
Indie Pop             1
Maskandi              1
Name: count, dtype: int64

In [110]:
# create manual groupings
genre_map = {'Hip Hop': 'Hip Hop', 'Pop Rap': 'Hip Hop',
    'Pop': 'Pop', 'Indie Pop': 'Pop', 'Indie-Pop': 'Pop', 'K-Pop': 'Pop', 'Q-Pop': 'Pop', 'Indonesian Pop': 'Pop', 'T-Pop': 'Pop', 
    'Cantopop': 'Pop', 'Alternative Pop': 'Pop',
    'Reggaeton': 'Latin', 'Latin Ballad': 'Latin', 'Regional Mexicano': 'Latin', 'Latin Urban': 'Latin',
    'Rock': 'Rock', 'J-Rock': 'Rock', 'Alternative Rock': 'Rock',
    'Filmi': 'Region Genre', 'Maskandi': 'Region Genre', 'Chalga': 'Region Genre', 'French': 'Region Genre',
    'Afrobeats': 'Other', 'Funk': 'Other', 'Jazz': 'Other', 'Psytrance': 'Other', 'Neo Soul': 'Other', 'Lo-Fi Hip Hop': 'Other', 
    'Indietronica': 'Other', 'Singer-Songwriter': 'Other'
}
             
                
genre_df_first['genre_main'] = genre_df_first['artist_genre'].map(genre_map)
#genre_df_first

In [111]:
tmp = genre_df_first.groupby('country')['genre_main'].agg(lambda x: x.value_counts().index[0]).reset_index()
tmp2 = genre_df_first.groupby('country')['artist_genre'].agg(lambda x: ", ".join(x.unique()[:5])).reset_index()
tmp3 = tmp.merge(tmp2, on='country')

In [112]:
import geopandas as gpd
import numpy as np

In [113]:
# world map with data
url = 'https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/world-countries.json'
world_url = gpd.read_file(url)


In [114]:
# merge df with geodata
world_merged = world_url.merge(tmp3, left_on='name', right_on='country', how='left')
#world_merged

In [115]:
# fix US
world_merged.loc[world_merged['name'] == 'United States of America', 'country'] = 'United States'
world_merged.loc[world_merged['name'] == 'United States of America', 'genre_main'] = 'Latin'
world_merged.loc[world_merged['name'] == 'United States of America', 'artist_genre'] = 'Reggaeton'

In [116]:
world_merged = world_merged.replace({np.nan: None})

In [117]:
# colors 
color_map = {'Hip Hop': '#ff5959',
             'Pop': '#00a1ff',
             'Latin': '#ddff61',
             'Rock': '#00ff00',
             'Region Genre': '#ffa500',
             'Other': '#00ffff'           
}

In [118]:
# plot genre maps, used seaborn because the text seemed cleaner than folium
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

colors = world_merged["genre_main"].map(lambda x: color_map.get(x, '#ffffff'))


# we'll use a function to get cleaner images for continent splits
def plot_genre_map(title, x_limits, y_limits, save):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # plot
    world_merged.plot(
        ax=ax,
        color=colors,
        edgecolor="black",
        linewidth=0.5
    )
    
    # text
    for _, country in world_merged.iterrows():
        if country["geometry"] is not None and country["genre_main"] not in ["Unknown", None]:
            point = country["geometry"].representative_point()
            
            # Only label if the point is within our current zoom window
            if x_limits[0] <= point.x <= x_limits[1] and y_limits[0] <= point.y <= y_limits[1]:
                label_text = str(country["artist_genre"]).replace(", ", "\n")
                ax.text(point.x, point.y, label_text,
                    fontsize=6,ha="center",va="center")


    # legend
    legend_handles = [
        mpatches.Patch(color=color, label=genre) 
        for genre, color in color_map.items()
    ]
    
    # Add the legend to the plot
    ax.legend(
        handles=legend_handles, 
        title="Genre Groups", 
        loc='lower left', # You can change to 'lower right' if it blocks data
        bbox_to_anchor=(0.05, 0.05), # Fine-tune position
        fontsize=10,
        frameon=True,
        facecolor='white',
        edgecolor='grey'
    )
    
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    ax.set_title(title, fontsize=15)
    ax.axis('off') 
    
    plt.tight_layout()
    plt.savefig(save)
    plt.show()




In [ ]:
# americas
plot_genre_map("", (-150, -30), (-57, 65), "map_americas.png")

In [ ]:
# europe
plot_genre_map("", (-20, 57), (-35, 75), "map_europe_africa.png")

In [ ]:
# asia
plot_genre_map("", (50, 152), (-40, 90), "map_asia.png")